In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import polars as pl

In [ ]:
output_dir = Path(r"Q:\Model Projects\Big Data Working Group\validation_data\BATS-2023")

In [ ]:
counties = gpd.read_file(
    r"Q:\GIS\Administrative\Census_Boundaries\TIGER\Census_County\tl_2020_06_county20.zip",
    columns=["GEOID20", "NAME20"],
)
pumas = gpd.read_file(
    r"Q:\GIS\Administrative\Census_Boundaries\TIGER\Census_PUMA\2020\tl_2025_06_puma20.zip",
    columns=["GEOID20", "NAMELSAD20"],
)

In [ ]:
def read_trips(filepath):
    trips = pd.read_csv(
        filepath,
        dtype={"oxco": float, "oyco": float, "dxco": float, "dyco": float},
    )
    trips["origin"] = gpd.points_from_xy(trips["oxco"], trips["oyco"])
    trips["destination"] = gpd.points_from_xy(trips["dxco"], trips["dyco"])
    return trips

In [ ]:
dir = Path(
    r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\tdm_market_analysis"
)
persons = pl.read_csv(dir / "person.csv").select(
    "hhno", "pno", adult=pl.col("pagey") > 17
)

In [ ]:
trips_gdf = (
    gpd.GeoDataFrame(
        read_trips(dir / "trip.csv"),
        geometry="origin",
        # BATS CRS assumed to be EPSG:4326 (WGS 1984) since it's GPS data
        crs="EPSG:4326",
    )
    .to_crs("EPSG:4269")
    .sjoin(
        counties,
        how="left",
    )
    .drop(columns="index_right")
    .rename(columns={"GEOID20": "o_county_geoid", "NAME20": "o_county"})
    .sjoin(
        pumas,
        how="left",
    )
    .drop(columns="index_right")
    .rename(columns={"GEOID20": "o_puma_geoid", "NAMELSAD20": "o_puma"})
    .set_geometry(
        "destination",
        # BATS CRS assumed to be EPSG:4326 (WGS 1984) since it's GPS data
        crs="EPSG:4326",
    )
    .to_crs("EPSG:4269")
    .sjoin(
        counties,
        how="left",
    )
    .drop(columns="index_right")
    .rename(columns={"GEOID20": "d_county_geoid", "NAME20": "d_county"})
    .sjoin(
        pumas,
        how="left",
    )
    .drop(columns="index_right")
    .rename(columns={"GEOID20": "d_puma_geoid", "NAMELSAD20": "d_puma"})
)

In [ ]:
trips = pl.from_pandas(
    pd.DataFrame(trips_gdf).drop(columns=["origin", "destination"])
).join(persons, on=["hhno", "pno"], how="left")
# trips.write_parquet(output_dir / "bats2023-trips.parquet")
# trips = pl.read_parquet(output_dir / "bats2023-trips.parquet")

In [ ]:
def group_trips(
    trips: pl.DataFrame,
    geog_cols: list[str],
    mode_col: str | None = None,
    private_vehicle_trips=False,
):
    bay_area_counties = [
        "San Francisco",
        "Marin",
        "Sonoma",
        "Napa",
        "Solano",
        "Contra Costa",
        "Alameda",
        "Santa Clara",
        "San Mateo",
    ]
    if mode_col is not None:
        if private_vehicle_trips:
            raise RuntimeError(
                "mode_col should be left as None if aggregating for private vehicle trips."
            )
        group_by_cols = geog_cols + ["adult", mode_col, "dpurp", "dept_hr"]
    else:
        group_by_cols = geog_cols + ["adult", "dpurp", "dept_hr"]
    filters = pl.col("o_county").is_in(bay_area_counties) & pl.col("d_county").is_in(
        bay_area_counties
    )
    with_cols = {"dept_hr": pl.col("deptm") // 100}
    agg_cols = [pl.len(), pl.sum("trexpfac")]
    if private_vehicle_trips:
        filters &= pl.col("mode").is_in([3, 4, 5])  # SOV, HOV2, HOV3+
        occupancy_factor = {
            # mode: divide by this factor
            3: 1.0,  # SOV
            4: 2.0,  # HOV2
            5: 3.5,  # HOV3+
        }
        with_cols |= {
            # to get private vehicle counts, scale HOV3+ by 1/3.5 and HOV2 by 1/2
            "veh_trexpfac": pl.col("trexpfac")
            / pl.col("mode").replace_strict(occupancy_factor),
        }
        agg_cols.append(pl.sum("veh_trexpfac"))
    return (
        trips.filter(filters)
        .with_columns(**with_cols)
        .group_by(group_by_cols)
        .agg(agg_cols)
        .sort(group_by_cols)
    )

In [ ]:
county_cols = ["o_county_geoid", "o_county", "d_county_geoid", "d_county"]
puma_o_cols = ["o_puma_geoid", "o_puma"]
puma_d_cols = ["d_puma_geoid", "d_puma"]

In [ ]:
# mode = Daysim modes
group_trips(trips, county_cols, "mode").write_csv(
    output_dir / "bats2023-trips-by_county-daysim_modes.csv"
)
group_trips(trips, puma_o_cols, "mode").write_csv(
    output_dir / "bats2023-trips-by_puma_origin-daysim_modes.csv"
)
group_trips(trips, puma_d_cols, "mode").write_csv(
    output_dir / "bats2023-trips-by_puma_destination-daysim_modes.csv"
)

In [ ]:
# mode_type = from BATS raw data (from codebook below):
# mode_type	1	Walk
# mode_type	10	School bus
# mode_type	11	Shuttle/vanpool
# mode_type	12	Ferry
# mode_type	13	Transit
# mode_type	14	Long distance passenger
# mode_type	2	Bike
# mode_type	3	Bikeshare
# mode_type	4	Scootershare
# mode_type	5	Taxi
# mode_type	6	TNC
# mode_type	7	Other
# mode_type	8	Car
# mode_type	9	Carshare
# mode_type	995	Missing Response
group_trips(trips, county_cols, "mode_type").write_csv(
    output_dir / "bats2023-trips-by_county-rsg_modes.csv"
)
group_trips(trips, puma_o_cols, "mode_type").write_csv(
    output_dir / "bats2023-trips-by_puma_origin-rsg_modes.csv"
)
group_trips(trips, puma_d_cols, "mode_type").write_csv(
    output_dir / "bats2023-trips-by_puma_destination-rsg_modes.csv"
)

In [ ]:
# private vehicle trips
group_trips(
    trips,
    county_cols,
    private_vehicle_trips=True,
).write_csv(output_dir / "bats2023-private_veh_trips-by_county.csv")
group_trips(
    trips,
    puma_o_cols,
    private_vehicle_trips=True,
).write_csv(output_dir / "bats2023-private_veh_trips-by_puma_origin.csv")
group_trips(
    trips,
    puma_d_cols,
    private_vehicle_trips=True,
).write_csv(output_dir / "bats2023-private_veh_trips-by_puma_destination.csv")